In [2]:
import numpy as np
import glob, os
import matplotlib.pyplot as plt
from datetime import datetime
from copy import deepcopy

In [3]:
base_dict = {"VAE_RMSD": None,
             "VAE_LOSS_RMSD": None,
             "PCA_RMSD": None,
             "PCA_LOSS_RMSD": None}

def load_files(numpy_files, verbose=True):
    fns = sorted(numpy_files)
    attrs = [fn.split('/')[1:] if fn.split('/')[0] == 'numpy_backups' else fn.split('/') for fn in fns]
    model_name = attrs[0][0]
    assert False not in [elem[0] == model_name for elem in attrs]
    
    model_data = {}
    for attr_set in attrs:
        n_latents = int(attr_set[1].split('_')[0])
        if n_latents not in model_data.keys():
            model_data[n_latents] = {}
        rpt = attr_set[2]
        if rpt not in model_data[n_latents].keys():
            model_data[n_latents][rpt] = deepcopy(base_dict)

        try:
            if attr_set[-1].split('.')[0] in [key for key in base_dict.keys()]:
                model_data[n_latents][rpt][attr_set[-1].split('.')[0]] = np.load(os.path.join(*attr_set))
            else:
                if verbose:
                    print(f"Excluding ", os.path.join(*attr_set))
        except:
            if verbose:
                    print(f"Failure on", os.path.join(*attr_set))
            
    return model_data

def lowest_rpts(chart_data):
    means = {}

    for key, val in chart_data.items():
        if key not in means.keys():
            means[key] = {}
        for key2, val2 in chart_data[key].items():
            if key2 not in means[key].keys():
                means[key][key2] = {}
            for key3, val3 in chart_data[key][key2].items():
                means[key][key2][key3] = np.mean(chart_data[key][key2][key3])
    
    best_rpts_RMSD = {key: None for key in means.keys()}
    best_rpts_LOSS = {key: None for key in means.keys()}
    
    for key, val in means.items():
        #discover the lowest mean rpt
        best_rpts_RMSD[key] = min(val.items(), key=lambda x: x[-1]['VAE_RMSD'])[0]
        best_rpts_LOSS[key] = min(val.items(), key=lambda x: x[-1]['VAE_LOSS_RMSD'])[0]
    
    return means, best_rpts_RMSD, best_rpts_LOSS

In [4]:
#Plot Violin of the best

def violin_plots(vae_dat, pca_dat, latents, title, ceil=10.0, num_stds=3, show=False, figure_dir=None, test_hists=False):
    import matplotlib.pyplot as plt
    figure_log = f"Began {datetime.now()} \n"
    figure_log += f"Log for making figure {title=} \n"
    
    #rmsds and pca rmsds are a dict of results with keys n_latent and vals hist_vals
    rpt_rmsds = [10*vae_dat[key] for key in vae_dat.keys()] #convert to angstrom
    rpt_pca_rmsds = [10*pca_dat[key] for key in pca_dat.keys()] #convert to angstrom
    og_shapes, og_pca_shapes = [data.shape[0] for data in rpt_rmsds], [data.shape[0] for data in rpt_pca_rmsds]
    
    #remove nan values
    rpt_rmsds = [data[np.where(~np.isnan(data))] for data in rpt_rmsds]
    rpt_pca_rmsds = [data[np.where(~np.isnan(data))] for data in rpt_pca_rmsds]
    non_nan_shapes, non_nan_pca_shapes = [data.shape[0] for data in rpt_rmsds], [data.shape[0] for data in rpt_pca_rmsds]

    #Report the number of Nan values, and remove datasets with all Nan values
    indices_all_nan = []
    for i, (latent, old1, old2, new1, new2) in enumerate(zip(latents, og_shapes, og_pca_shapes, non_nan_shapes, non_nan_pca_shapes)):
        figure_log += f"Analyze NAN content for {latent} Latents \n"
        figure_log += f"For model with {latent} Latents, \n \t {old1-new1} Nan values were found in VAE \n \t {old2 - new2} Nan values were found in PCA \n"
        if new1 == 0:
            figure_log += f"For model with {latent} Latents, \n \t All values are NAN - a dashed line on the plot \n"
            indices_all_nan.append(i)

    #Now remove values above a ridiculous threshold, like 10 angstroms
    rpt_rmsds = [data[np.where(data < ceil)] for data in rpt_rmsds]
    rpt_pca_rmsds = [data[np.where(data < ceil)] for data in rpt_pca_rmsds]
    in_thresh_shapes, in_thresh_pca_shapes = [data.shape[0] for data in rpt_rmsds], [data.shape[0] for data in rpt_pca_rmsds]

    #Report the number of values above the threshold
    for i, (latent, old1, old2, new1, new2) in enumerate(zip(latents, non_nan_shapes, non_nan_pca_shapes, in_thresh_shapes, in_thresh_pca_shapes)):
        figure_log += f"Analyze Outliers content for {latent} Latents \n"
        figure_log += f"For model with {latent} Latents, \n \t {old1-new1} values were found in VAE above the ceiling of {ceil} Angstroms \n \t {old2 - new2} values were found in PCA above the ceiling of {ceil} Angstroms \n"

    #Retrim - all datasets should be composed of real values between 0 and ceil, remove empty sets now (if they were all nan essentially)
    rpt_rmsds = [data for i, data in enumerate(rpt_rmsds) if i not in indices_all_nan]
    rpt_pca_rmsds = [data for i, data in enumerate(rpt_pca_rmsds) if i not in indices_all_nan]
    latents2plot = [data for i, data in enumerate(latents) if i not in indices_all_nan]
    
    #detect outliers - calculate median, std, and do not plot above n stds
    rpt_rmsd_medians = [np.median(arr) for arr in rpt_rmsds]
    rpt_rmsd_stds = [np.std(arr) for arr in rpt_rmsds]
    rpt_pca_rmsd_medians = [np.median(arr) for arr in rpt_pca_rmsds]
    rpt_pca_rmsd_stds = [np.std(arr) for arr in rpt_pca_rmsds]
    rpt_maxima = [median + 3*std for median, std in zip(rpt_rmsd_medians, rpt_rmsd_stds)]
    rpt_pca_maxima = [median + 3*std for median, std in zip(rpt_pca_rmsd_medians, rpt_pca_rmsd_stds)]

    figure_log += f"Report the median and standard deviation used for each Latent \n  \t VAE \n"
    for elem in [f"\t\t L-{latent} Med-{median:0.3f} Std-{std:0.3f} \n" for latent, median, std in zip(latents2plot, rpt_rmsd_medians, rpt_rmsd_stds)]:
        figure_log += elem
    figure_log += f"  \t PCA \n"
    for elem in [f"\t\t L-{latent} Med-{median:0.3f} Std-{std:0.3f} \n" for latent, median, std in zip(latents2plot, rpt_pca_rmsd_medians, rpt_pca_rmsd_stds)]:
        figure_log += elem
    figure_log += '\n'
    
    #Reduce charting data to those within the threshold
    rpt_data2plot = [rpt_rmsd[np.where(rpt_rmsd < maximum)] for rpt_rmsd, maximum in zip(rpt_rmsds, rpt_maxima)]
    rpt_pca_data2plot = [rpt_pca_rmsd[np.where(rpt_pca_rmsd < maximum)] for rpt_pca_rmsd, maximum in zip(rpt_pca_rmsds, rpt_pca_maxima)]
    
    rpt_data2exclude = [np.where(rpt_rmsd > maximum)[0].shape[0] for rpt_rmsd, maximum in zip(rpt_rmsds, rpt_maxima)]
    rpt_pca_data2exclude = [np.where(rpt_pca_rmsd > maximum)[0].shape[0] for rpt_pca_rmsd, maximum in zip(rpt_pca_rmsds, rpt_pca_maxima)]

    figure_log += (f"Number of points that are {num_stds:0.3f} stds above the median \n \t VAE \n")
    for elem in [f"\t\t L-{latent} Num-{elem} \n" for latent, elem in zip(latents2plot, rpt_data2exclude)]:
        figure_log += elem
    figure_log += (f" \t PCA \n")
    for elem in [f"\t\t L-{latent} Num-{elem} \n" for latent, elem in zip(latents2plot, rpt_pca_data2exclude)]:
        figure_log += elem
    figure_log += '\n'

    figure_log += (f"Final number of points comprising violin \n \t VAE \n")
    for elem in [f"\t\t L-{latent} Num-{elem.shape[0]} \n" for latent, elem in zip(latents2plot, rpt_data2plot)]:
        figure_log += elem
    figure_log += (f" \t PCA \n")
    for elem in [f"\t\t L-{latent} Num-{elem.shape[0]} \n" for latent, elem in zip(latents2plot, rpt_pca_data2plot)]:
        figure_log += elem
    figure_log += '\n'
    
    plt.clf()
    fig = plt.figure(figsize=(3.25, 3.25))
    #Make the domain linear on n_latents = 1, 2, 3, 4 and log for 8, 16 etc
    domain = [n-0.05 if n <= 4 else 2+np.log2(n)-0.05 for n in latents2plot]
    v1 = plt.violinplot(rpt_data2plot, domain, showextrema=False)
    domain = [n+0.05 if n <= 4 else 2+np.log2(n)+0.05 for n in latents2plot]
    v2 = plt.violinplot(rpt_pca_data2plot, domain, showextrema=False)

    for i, b in enumerate(v1['bodies']):
        b.set_edgecolor('black')
        b.set_alpha(1)
        # get the center (the center is actually the domain point)
        m = np.mean(b.get_paths()[0].vertices[:, 0])
        # modify the paths to not go further right than the center
        b.get_paths()[0].vertices[:, 0] = np.clip(b.get_paths()[0].vertices[:, 0], -np.inf, m)
        b.set_color('r')
        
    for b in v2['bodies']:
        b.set_edgecolor('black')
        b.set_alpha(1)
        # get the center
        m = np.mean(b.get_paths()[0].vertices[:, 0])
        # modify the paths to not go further left than the center
        b.get_paths()[0].vertices[:, 0] = np.clip(b.get_paths()[0].vertices[:, 0], m, np.inf)
        b.set_color('b')
    
    #add some verticle lines where we excluded the NAN values, min and max set by the plotted data
    global_min = np.min([np.min(dataset) for dataset in rpt_data2plot])
    global_max = np.max([np.max(dataset) for dataset in rpt_data2plot])
    latents2vline = [n for n in latents if n not in latents2plot]
    latents2vline = [n-0.05 if n <= 4 else 2.05+np.log2(n) for n in latents2vline]
    
    plt.vlines(latents2vline, ymin=global_min, ymax=global_max,
              colors='red', linestyles='dashed')
    
    plt.xlabel('N_Latents')
    plt.ylabel('Reconstruction Error (Angstrom)')
    plt.title(title)
    #rotation = [0,0,0,0] + [45]*(len(latents)-4)
    plt.xticks(ticks=np.arange(1, len(latents)),
               labels=[''+str(latent) for latent in latents[:-1]],
               rotation='vertical', ha='left')
    
    plt.savefig(os.path.join(figure_dir, f'reconstruction_rmsd_{title=}.png'), bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    
    with open(os.path.join(figure_dir, f'figure_generation_log.txt'), 'w') as f:
        figure_log += f"Successfully generated and saved figure and log by {datetime.now()} \n"
        f.write(figure_log)
    
    return fig, figure_log

In [5]:
def title_of_model(name):
    if name.startswith('OX_X'):
        return 'Oxycodone'
    elif name.startswith('DA_X'):
        return 'Deca-Alanine'
    elif name.startswith('DA_stretch'):
        return 'Deca-Alanine Helix Stretch'
    elif name.startswith('CR_X'):
        return 'Crambin'
    elif name.startswith('BR_X'):
        return 'BRD4/JQ1'
    elif name.startswith('HIV1p_X'):
        return 'HIV1-Protease'
    elif name.startswith('KOR_X'):
        return 'KOR/ak'
    else:
        raise Exception(f'No title found for {name}!')

In [6]:
base_models, model_prefixes = [f'X013-{i}' for i in [1, 2, 3, 4]], ['OX_', 'DA_', 'DA_stretch_', 'CR_', 'BR_', 'HIV1p_']

In [7]:
#base_models, model_prefixes = ['X013-2', 'X013-4'], ['CR_']

In [9]:
def print_lat(i):
    """
    Convert expected index to expected latent (not heavy atom quantity)
    """
    if i in [0,1,2,3]:
        return i + 1
    else:
        return 2**(i-1)

show = False

for base_model in base_models:
    f = open(f"best_model_list_{base_model}.txt", 'w')
    for model_pre in model_prefixes:
        model_name2load = model_pre + base_model
        print(model_name2load)
        chart_data = load_files(glob.glob(f'numpy_backups/{model_name2load}/*/*/*.npy'), verbose=False)
        means, best_rpts_rmsd, best_rpts_loss = lowest_rpts(chart_data)
        for i, rpt in enumerate(best_rpts_rmsd.values()):
            json_fn = f"json_inputs/{base_model}/{model_name2load}/{model_name2load}_{print_lat(i):04d}_{int(rpt[-1]):02d}.json\n"
            f.write(json_fn)
    f.close()
    print(base_model)
        #print(best_rpts_loss)

OX_X013-1
DA_X013-1
DA_stretch_X013-1
CR_X013-1
BR_X013-1
HIV1p_X013-1
X013-1
OX_X013-2
DA_X013-2
DA_stretch_X013-2
CR_X013-2
BR_X013-2
HIV1p_X013-2
X013-2
OX_X013-3
DA_X013-3
DA_stretch_X013-3
CR_X013-3
BR_X013-3
HIV1p_X013-3
X013-3
OX_X013-4
DA_X013-4
DA_stretch_X013-4
CR_X013-4
BR_X013-4
HIV1p_X013-4
X013-4


In [ ]:
show = False

for base_model in base_models:
    for model_pre in model_prefixes:
        model_name2load = model_pre + base_model
        print(model_name2load)
        chart_data = load_files(glob.glob(f'numpy_backups/{model_name2load}/*/*/*.npy'), verbose=False)
        means, best_rpts_rmsd, best_rpts_loss = lowest_rpts(chart_data)
        
        vae_rmsds =      {key: val[best_rpts_rmsd[key]]['VAE_RMSD']      for key, val in chart_data.items()}
        vae_loss_rmsds = {key: val[best_rpts_loss[key]]['VAE_LOSS_RMSD'] for key, val in chart_data.items()}
        pca_rmsds =      {key: val[best_rpts_rmsd[key]]['PCA_RMSD']      for key, val in chart_data.items()}
        pca_loss_rmsds = {key: val[best_rpts_loss[key]]['PCA_LOSS_RMSD'] for key, val in chart_data.items()}
        
        latents = [int(key) for key in vae_rmsds]
    
        figure_dir = os.path.join(os.getcwd(), f'figures/{model_name2load}/')
        if not os.path.isdir(figure_dir):
            os.makedirs(figure_dir, exist_ok=True)
        
        _ = violin_plots(vae_rmsds, pca_rmsds, latents, model_name2load,
                         ceil=10.0, num_stds=3, show=show, figure_dir=figure_dir)
print("Done!")

In [10]:
def farlier_part(lines):
    latents, vae_nans, pca_nans = [], [], []
    for line in lines:
        if line.startswith('Analyze Outliers content for'):
            num_latents = line.split(' ')[4]
            index_this_line = lines.index(line)
            #print(index_this_line, num_latents)
            vae_line, pca_line = lines[index_this_line + 2], lines[index_this_line + 3]
            latents.append(num_latents)
            vae_nans.append([elem for elem in vae_line.split(' ') if elem][1])
            pca_nans.append([elem for elem in pca_line.split(' ') if elem][1])
    return latents, vae_nans, pca_nans


def NAN_part(lines):
    latents, vae_nans, pca_nans = [], [], []
    for line in lines:
        if line.startswith('Analyze NAN content for'):
            num_latents = line.split(' ')[4]
            index_this_line = lines.index(line)
            #print(index_this_line, num_latents)
            vae_line, pca_line = lines[index_this_line + 2], lines[index_this_line + 3]
            latents.append(num_latents)
            vae_nans.append([elem for elem in vae_line.split(' ') if elem][1])
            pca_nans.append([elem for elem in pca_line.split(' ') if elem][1])
    return latents, vae_nans, pca_nans


def mean_std_part(lines):
    start_line_index = lines.index([line for line in lines if line.startswith("Report the median and standard deviation")][0])
    end_line_index = lines.index([line for line in lines if line.startswith("Number of points that are ")][0]) - 1
    lines = lines[start_line_index+1:end_line_index]
    halfway = len(lines) // 2
    vae_lines, pca_lines = lines[1:halfway], lines[halfway+1:]

    vae_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in vae_lines]
    vae_meds = [[elem for elem in line.split(' ') if elem.startswith('Med-')][0][4:] for line in vae_lines]
    vae_stds = [[elem for elem in line.split(' ') if elem.startswith('Std-')][0][4:] for line in vae_lines]
    pca_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in pca_lines]
    pca_meds = [[elem for elem in line.split(' ') if elem.startswith('Med-')][0][4:] for line in pca_lines]
    pca_stds = [[elem for elem in line.split(' ') if elem.startswith('Std-')][0][4:] for line in pca_lines]
    #Same Latents, same length of everything
    assert vae_lats == pca_lats and len(vae_meds) == len(vae_stds) and \
           len(pca_meds) == len(pca_stds) and len(vae_lats) == len(vae_meds) and len(vae_meds) == len(vae_stds)

    return vae_lats, vae_meds, vae_stds, pca_meds, pca_stds

def num_above_mean_std_part(lines):
    start_line_index = lines.index([line for line in lines if line.startswith("Number of points that are ")][0])
    end_line_index = lines.index([line for line in lines if line.startswith("Final number of points comprising violin")][0]) - 1
    lines = lines[start_line_index+1:end_line_index]
    halfway = len(lines) // 2
    vae_lines, pca_lines = lines[1:halfway], lines[halfway+1:]

    vae_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in vae_lines]
    vae_nums = [[elem for elem in line.split(' ') if elem.startswith('Num-')][0][4:] for line in vae_lines]
    
    pca_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in pca_lines]
    pca_nums = [[elem for elem in line.split(' ') if elem.startswith('Num-')][0][4:] for line in pca_lines]

    #Same latents, same length of everything
    assert vae_lats == pca_lats and len(vae_nums) == len(vae_lats) and len(pca_nums) == len(vae_nums)

    return vae_lats, vae_nums, pca_nums

def num_in_violin(lines):
    start_line_index = lines.index([line for line in lines if line.startswith("Final number of points comprising violin")][0])
    end_line_index = lines.index([line for line in lines if line.startswith("Successfully generated and saved figure")][0]) - 1
    lines = lines[start_line_index+1:end_line_index]
    halfway = len(lines) // 2
    vae_lines, pca_lines = lines[1:halfway], lines[halfway+1:]

    vae_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in vae_lines]
    vae_nums = [[elem for elem in line.split(' ') if elem.startswith('Num-')][0][4:] for line in vae_lines]
    
    pca_lats = [[elem for elem in line.split(' ') if elem.startswith('L-')][0][2:] for line in pca_lines]
    pca_nums = [[elem for elem in line.split(' ') if elem.startswith('Num-')][0][4:] for line in pca_lines]

    #Same latents, same length of everything
    assert vae_lats == pca_lats and len(vae_nums) == len(vae_lats) and len(pca_nums) == len(vae_nums)

    return vae_lats, vae_nums, pca_nums

def figure_log_to_csv(figure_log_fn):
    """
    Convert the figure log string to a csv string

    Columns:
        NAN - Number of particles that were NAN on passthrough
        > 10 A - had reconstruction error greater than 1 nanometer
        Median (Ang) - median of the violin distribution reported in Angstrom
        Std (Ang) - standard deviation of the violin distribution in Angstrom
        N Omit - Number of particles that are greater than n standard deviations from median (default n=3)
        N Violin - Number of particles that are present in the violin histogram

    Each column should be generated for each latent and each method (for latent in latents: for method in VAE PCA:)
    """
    with open(figure_log_fn, 'r') as f:
        lines = [line for line in f.readlines()]
    name = [line for line in lines if 'title' in line][0][29:-3]
    file_contents = f",,{name},,,,,,\n"
    cols = ["NAN", "> 10 A", "Median (Ang)", "Std (Ang)", "N Omit", "N Violin"]
    file_contents += f"Latents,,"+ ','.join(cols) + '\n'

    latents, vae_nan, pca_nan = NAN_part(lines)
    _, vae_farlier, pca_farlier = farlier_part(lines)
    latents_plotted, vae_meds, vae_stds, pca_meds, pca_stds = mean_std_part(lines)
    _, vae_outlier, pca_outlier = num_above_mean_std_part(lines)
    _, vae_violin, pca_violin = num_in_violin(lines)

    vae_lines = []
    for i, latent in enumerate(latents):
        num_nan, num_far = vae_nan[i], vae_farlier[i]
        if latent in latents_plotted:
            i = latents_plotted.index(latent)
            median, std, num_outlier, num_violin = vae_meds[i], vae_stds[i], vae_outlier[i], vae_violin[i]
        else:
            median, std, num_outlier, num_violin = 0, 0, 0, 0
        vae_lines.append(f"{latent},VAE (red),{num_nan},{num_far},{median},{std},{num_outlier},{num_violin}\n")
    
    pca_lines = []
    for i, latent in enumerate(latents):
        num_nan, num_far = pca_nan[i], pca_farlier[i]
        if latent in latents_plotted:
            i = latents_plotted.index(latent)
            median, std, num_outlier, num_violin = pca_meds[i], pca_stds[i], pca_outlier[i], pca_violin[i]
        else:
            median, std, num_outlier, num_violin = 0, 0, 0, 0
        pca_lines.append(f"{latent},PCA (blue),{num_nan},{num_far},{median},{std},{num_outlier},{num_violin}\n")

    assert len(vae_lines) == len(pca_lines)
    
    for i, line in enumerate(vae_lines):
        file_contents += line
        file_contents += pca_lines[i]
        
    return file_contents

In [11]:
figure_generation_logs = sorted(glob.glob('./figures/*/figure_generation_log.txt'))

In [12]:
csv_direc = './tables/csvs_violin/'

for log_fn in figure_generation_logs:
    csv_name = os.path.join(csv_direc, log_fn.split('/')[-2] + '_violin_health_report.csv')
    csv_contents = figure_log_to_csv(log_fn)
    with open(csv_name, 'w') as f:
        f.write(csv_contents)

In [22]:
def reorganize_figures_and_logs(figure_dir):
    import glob, shutil
    unique_models = []
    for content in os.listdir(figure_dir):
        if content.startswith('X') or '_' not in content:
            continue
        name_parts = content.split('_')
        if name_parts[-1].startswith('X') and name_parts[-1] not in unique_models:
            unique_models.append(name_parts[-1])
    #print(unique_models)
    for model_name in sorted(unique_models):
        if not os.path.isdir(os.path.join(figure_dir, model_name)):
            os.makedirs(os.path.join(figure_dir, model_name))
        for content in os.listdir(figure_dir):
            if content.startswith('X') or '_' not in content:
                continue
            name_parts = content.split('_')
            if name_parts[-1] == model_name:
                pngs = sorted(glob.glob(os.path.join(figure_dir, content, '*.png')))
                #print(pngs)
                logs = sorted(glob.glob(os.path.join(figure_dir, content, 'figure_generation_log.txt')))
                #print(logs)
                for png, log in zip(pngs, logs):
                    shutil.copy(png, os.path.join(figure_dir, model_name, os.path.basename(png)))
                    shutil.copy(log, os.path.join(figure_dir, model_name, os.path.basename(png)[:-4]+"_"+os.path.basename(log)))
                    

In [23]:
reorganize_figures_and_logs('./figures/')